In [10]:
import asyncio
import av
from tqdm import tqdm
import io
from urllib.parse import urlparse
from PIL import Image


def parse_s3_url(s3_url: str) -> tuple[str, str]:
    parsed = urlparse(s3_url)
    return parsed.netloc, parsed.path.lstrip("/")

class FastFrameReader:
    """High-performance random-access frame extractor using PyAV FFmpeg bindings."""

    def __init__(self, video_bytes: bytes):
        self.video_bytes = video_bytes
        self.buffer = io.BytesIO(video_bytes)
        self.container = av.open(self.buffer)
        self.stream = self.container.streams.video[0]
        self.fps = float(self.stream.average_rate)  # type: ignore
        if self.fps <= 0:
            raise RuntimeError("Invalid FPS detected in video.")


    def get_frame(self, frame_index: int) -> bytes:
        ts_seconds = frame_index / self.fps
        seek_ts = int(ts_seconds / self.stream.time_base)


        self.container.seek(seek_ts, stream=self.stream)#type:ignore

        for frame in self.container.decode(video=0):
            if frame.pts is None:
                continue

            ts_sec = frame.pts * self.stream.time_base
            frame_number = int(ts_sec * self.fps)

            if frame_number >= frame_index:
                return self._encode_webp(frame)

        raise RuntimeError(f"Could not decode requested frame {frame_index}.")

    def _encode_webp(self, frame: av.VideoFrame) -> bytes:
        rgb = frame.to_ndarray(format="rgb24")
        img = Image.fromarray(rgb)

        buf = io.BytesIO()
        img.save(buf, format="WEBP", quality=80)  
        return buf.getvalue()
    

def extract_frames(video_bytes: bytes, indices: list[int]) -> list[bytes]:
    reader = FastFrameReader(video_bytes)
    results = []
    for idx in tqdm(sorted(indices)):
        img_bytes = reader.get_frame(idx)
        results.append(img_bytes)
    return results

async def extract_frames_async(video_bytes: bytes, indices: list[int]) -> list[bytes]:
    loop = asyncio.get_running_loop()
    return await loop.run_in_executor(None, extract_frames, video_bytes, indices)



def get_segment_frame_indices(start: int, end: int, n: int) -> list[int]:
    if n <= 0 or end <= start:
        return []

    total = end - start
    return [start + (i + 1) * total // (n + 1) for i in range(n)]

In [11]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
ROOT_DIR = Path(os.getcwd()).parent
import asyncio
print(ROOT_DIR)
sys.path.insert(0, str(ROOT_DIR))
from core.storage import StorageClient
from urllib.parse import urlparse
from core.config.storage import MinioSettings

/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT/ingestion


In [12]:
settings = MinioSettings(
    host="localhost",
    port="9000",
    user="minioadmin",
    password="minioadmin"
)
minio_client = StorageClient(settings)


def parse_s3_url(s3_url: str) -> tuple[str, str]:
    parsed = urlparse(s3_url)
    return parsed.netloc, parsed.path.lstrip("/")


async def get_video_bytes(s3_url: str, storage: StorageClient):
    bucket, object_name = parse_s3_url(s3_url)
    loop = asyncio.get_running_loop()
    data = await loop.run_in_executor(None, lambda: storage.get_object(bucket, object_name))
    return data

In [13]:
local_video_bytes = await get_video_bytes(
    "s3://videotest/K01_V002.mp4",
    minio_client,
)
type(local_video_bytes)

bytes

In [14]:
# indices = [2, 4, 6, 22, 35, 48, 141, 219, 297, 394, 411, 428, 456, 465, 474, 493, 502, 511, 530, 539, 548, 568, 578, 588, 608, 617, 626, 645, 654, 663, 683, 692, 701, 720, 729, 738, 757, 766, 775, 794, 803, 812, 831, 840, 849, 869, 878, 887, 907, 916, 925, 944, 953, 962, 981, 990, 999, 1019, 1028, 1037, 1069, 1091, 1113, 1274, 1412, 1550, 1717, 1744, 1771, 1817, 1836, 1855, 1895, 1916, 1937, 1983, 2007, 2031, 2092, 2127, 2162, 2234, 2271, 2308, 2386, 2427, 2468, 2547, 2585, 2622, 2688, 2715, 2742, 2815, 2859, 2903, 2993, 3038, 3083, 3151, 3173, 3195, 3239, 3260, 3281, 3319, 3335, 3351, 3381, 3395, 3409, 3548, 3672, 3796, 3941, 3961, 3981, 4036, 4071, 4106, 4169, 4197, 4224, 4276, 4299, 4322, 4371, 4396, 4420, 4471, 4496, 4521, 4574, 4600, 4626, 4679, 4704, 4729, 4777, 4799, 4821, 4867, 4891, 4915, 4961, 4983, 5005, 5050, 5072, 5094, 5138, 5159, 5179, 5282, 5363, 5444, 5547, 5567, 5587, 5620, 5632, 5644, 5683, 5710, 5736, 5790, 5816, 5842, 5876, 5882, 5888, 5921, 5948, 5975, 6140, 6278, 6415, 6586, 6619, 6652, 6700, 6714, 6728, 6762, 6781, 6800, 6836, 6852, 6868, 6896, 6907, 6918, 6946, 6962, 6978, 7011, 7028, 7044, 7079, 7097, 7115, 7154, 7174, 7194, 7245, 7276, 7306, 7356, 7374, 7392, 7435, 7458, 7481, 7515, 7525, 7535, 7579, 7612, 7645, 7694, 7709, 7724, 7759, 7778, 7797, 7823, 7830, 7836, 7858, 7872, 7886, 7920, 7939, 7958, 7991, 8005, 8019, 8047, 8061, 8075, 8204, 8319, 8433, 8587, 8625, 8663, 8748, 8793, 8838, 8926, 8968, 9010, 9086, 9119, 9152, 9220, 9254, 9288, 9348, 9372, 9396, 9444, 9467, 9490, 9538, 9562, 9586, 9635, 9658, 9681, 9727, 9749, 9771, 9816, 9839, 9861, 9901, 9917, 9933, 9983, 10016, 10049, 10134, 10184, 10234, 10317, 10349, 10381, 10427, 10440, 10453, 10507, 10547, 10587, 10658, 10689, 10720, 10959, 11167, 11375, 11661, 11738, 11815, 11929, 11965, 12000, 12055, 12074, 12092, 12122, 12133, 12143, 12165, 12176, 12186, 12208, 12219, 12230, 12252, 12263, 12273, 12295, 12306, 12316, 12338, 12349, 12359, 12381, 12392, 12402, 12428, 12443, 12458, 12493, 12512, 12531, 12572, 12592, 12612, 12792, 12950, 13108, 13303, 13338, 13373, 13448, 13487, 13526, 13592, 13619, 13646, 13700, 13727, 13753, 13811, 13842, 13873, 13930, 13956, 13981, 14030, 14053, 14076, 14129, 14159, 14189, 14249, 14279, 14309, 14372, 14405, 14437, 14495, 14520, 14545, 14687, 14804, 14921, 15047, 15055, 15063, 15131, 15190, 15249, 15355, 15402, 15449, 15543, 15590, 15636, 15716, 15749, 15781, 15829, 15844, 15858, 15916, 15958, 16000, 16064, 16085, 16106, 16148, 16169, 16189, 16235, 16260, 16284, 16331, 16352, 16373, 16504, 16613, 16721, 16872, 16913, 16954, 17017, 17037, 17057, 17098, 17117, 17136, 17170, 17184, 17198, 17227, 17240, 17253, 17304, 17340, 17376, 17426, 17438, 17450, 17492, 17521, 17550, 17593, 17607, 17621, 17648, 17661, 17674, 17718, 17749, 17780, 17842, 17873, 17904, 17957, 17978, 17999, 18039, 18057, 18075, 18123, 18151, 18179, 18241, 18275, 18309, 18365, 18386, 18407, 18451, 18474, 18496, 18545, 18570, 18595, 18765, 18909, 19053, 19231, 19265, 19299, 19346, 19359, 19371, 19413, 19442, 19470, 19544, 19588, 19632, 19693, 19708, 19723, 19764, 19788, 19812, 19874, 19911, 19948, 20005, 20024, 20043, 20107, 20150, 20193, 20273, 20309, 20345, 20421, 20460, 20499, 20583, 20627, 20671, 20743, 20770, 20797, 20842, 20858, 20874, 20901, 20912, 20922, 20944, 20955, 20966, 20988, 20999, 21009, 21031, 21042, 21052, 21074, 21085, 21095, 21117, 21128, 21138, 21182, 21214, 21246, 21314, 21348, 21382, 21519, 21620, 21721, 21857, 21891, 21925, 21976, 21992, 22008, 22049, 22072, 22095, 22138, 22158, 22177, 22220, 22243, 22265, 22308, 22327, 22346, 22390, 22414, 22438, 22484, 22506, 22527, 22576, 22602, 22628, 22685, 22715, 22745, 22796, 22816, 22836, 22882, 22906, 22930, 22983, 23011, 23038, 23162, 23258, 23354, 23483, 23516, 23548, 23602, 23622, 23642, 23680, 23698, 23716, 23747, 23760, 23772, 23806, 23826, 23846, 23885, 23903, 23920, 23957, 23976, 23995, 24031, 24048, 24065, 24098, 24114, 24129, 24165, 24185, 24205, 24245, 24265, 24285, 24321, 24337, 24352, 24383, 24397, 24411, 24457, 24489, 24521, 24622, 24690, 24758, 24902, 24978, 25053, 25227, 25324, 25421, 25541, 25562, 25583, 25664, 25724, 25784, 25882, 25919, 25956, 26021, 26047, 26073, 26145, 26191, 26237, 26359, 26435, 26511, 26631, 26675, 26719, 26794, 26824, 26854, 26926, 26966, 27006, 27092, 27137, 27182, 27267, 27307, 27347, 27416, 27444, 27472, 27528, 27554, 27580, 27631, 27654, 27677, 27799, 27897, 27995, 28120, 28146, 28172, 28241, 28282, 28323, 28402, 28438, 28474, 28542, 28574, 28605, 28681, 28724, 28767, 28842, 28872, 28902, 28962, 28991, 29020, 29069, 29089, 29108, 29160, 29191, 29222, 29307, 29360, 29412, 29545, 29625, 29705, 29812, 29839, 29866, 29927, 29960, 29993, 30085, 30143, 30201, 30293, 30326, 30359, 30418, 30443, 30468, 30532, 30570, 30608, 30680, 30714, 30747, 30851, 30920, 30989, 31095, 31132, 31168, 31230, 31255, 31280, 31326, 31347, 31367, 31409, 31430, 31451, 31495, 31518, 31540, 31587, 31611, 31635, 31682, 31705, 31727, 31772, 31794, 31815, 31860, 31883, 31906, 31948, 31966, 31984, 32082, 32161, 32240, 32348, 32374]

indices = [
    2, 4, 6, 22, 35, 48, 141, 219, 297, 394, 411, 428, 456, 465, 474, 493, 502, 511, 530, 539, 548, 568, 578, 588, 608, 617, 626, 645, 654, 663, 683
]

In [15]:
frames = extract_frames(local_video_bytes, indices)

  0%|          | 0/31 [00:00<?, ?it/s]

100%|██████████| 31/31 [00:06<00:00,  5.06it/s]


In [16]:
from io import BytesIO
for idx, frame in enumerate(frames):
    minio_client.upload_fileobj(
        bucket="tmp",
        object_name=f"frame_{idx:08d}.webp",
        file_obj=BytesIO(frame),
        content_type="Image/webp"
    )

2025-11-15 14:24:45.695 | INFO     | core.storage:_ensure_bucket:45 - Bucket named: tmp does not exist, creating
2025-11-15 14:24:45.699 | INFO     | core.storage:upload_fileobj:85 - Uploaded object s3://tmp/frame_00000000.webp
2025-11-15 14:24:45.703 | INFO     | core.storage:upload_fileobj:85 - Uploaded object s3://tmp/frame_00000001.webp
2025-11-15 14:24:45.706 | INFO     | core.storage:upload_fileobj:85 - Uploaded object s3://tmp/frame_00000002.webp
2025-11-15 14:24:45.709 | INFO     | core.storage:upload_fileobj:85 - Uploaded object s3://tmp/frame_00000003.webp
2025-11-15 14:24:45.712 | INFO     | core.storage:upload_fileobj:85 - Uploaded object s3://tmp/frame_00000004.webp
2025-11-15 14:24:45.715 | INFO     | core.storage:upload_fileobj:85 - Uploaded object s3://tmp/frame_00000005.webp
2025-11-15 14:24:45.719 | INFO     | core.storage:upload_fileobj:85 - Uploaded object s3://tmp/frame_00000006.webp
2025-11-15 14:24:45.723 | INFO     | core.storage:upload_fileobj:85 - Uploaded obj

In [18]:
bucket = "tmp"
objects = minio_client.client.list_objects(bucket, recursive=True)
for obj in objects:
    minio_client.client.remove_object(bucket, obj.object_name)



minio_client.client.remove_bucket(bucket)